# SLM Zero Shot Model Instantiation 

This notebook is intended to test SLM model instantiation, prompting, and finetuning. Essentially "scrap notebook".


Notes:
- Investigate zero-shot vs few-shot learning
- Small ("Compact") Language model: 
    - SmolLM2


Libraries to install (?):
- `transformers`


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

messages = [
    {"role": "user", "content": "What is gravity?"}
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

enc = tokenizer(input_text, return_tensors="pt").to(device)

outputs = model.generate(
    **enc,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True
)

new_tokens = outputs[0][enc["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

# print("Prompt sent to model:\n", input_text)
# print("\nAssistant response:\n", response)
print(response)

Gravity is a fundamental force of nature that attracts objects with mass towards each other. It is a result of the interaction between mass, energy, and space. According to Newton's law of universal gravitation, every point mass attracts every other point mass by a force that is proportional to the product of their masses and inversely proportional to the square of the distance between them.

In simpler terms, gravity is the force that keeps objects from flying apart, and it is a result of the curvature of spacetime caused by the presence of mass and energy. This curvature is what we experience as gravity, and it is a fundamental aspect of the universe.

Think of it this way: if you were to drop an object from a height, it would fall towards the ground because the object's mass is causing it to curve the space around it. This is why objects fall towards the ground, and why we experience gravity.

Gravity is a powerful force that governs the behavior of objects in the universe, and it p

**To-Do list:**
- Understand Recipe-MPR better for fine tuning purposes
- Decide on fine-tuning strategy for SmolLM2; Prompt formatting and expected output will be important
        - LoRA is going to be important for training and WRITING.
- Set up wrapper for Baseline prompting of SLM


For future reference, we will probably eventually do something like:

```py
messages = [
    {
        "role": "user",
        "content": """User request: I want a quick vegetarian breakfast high in protein.

Options:
A. Bacon and egg sandwich
B. Spinach tofu scramble
C. Chicken breakfast burrito
D. Ham omelet
E. Sausage biscuits

Reply with only one letter: A, B, C, D, or E."""
    }
]
```



### Likely Strategy for fine-tuning:

```
<chat template>
user: I want a vegetarian breakfast high in protein

Options:
A bacon sandwich
B tofu scramble
C chicken burrito
D ham omelet
E sausage biscuits

Reply with only one letter.

assistant: B 
```

During training the model learns:

```given prompt → produce "B"```.